# 010 Feature Interactions

Цель: проверить комбинации фичей на 200ms.

Комбинации:
- imbalance_l1 + spread
- imbalance_l1 + microprice_diff
- imbalance_l1 + order_flow_imbalance (approx)

Используем 2D биннинг (квантили) → mean(target).

Структура:
1. Load data
2. Compute feature
3. Build target (200ms)
4. Basic statistics
5. Distribution plots
6. Relationship to future price (binning)
7. Time series (downsampled)
8. Correlation analysis
9. Extreme events analysis


In [ ]:
# 1. Load data
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = None
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "pyproject.toml").exists() and (p / "research").is_dir():
        sys.path.insert(0, str(p))
        root = p
        break

root = root or Path.cwd()
data_dir = root / "data" / "reconstructed"

In [ ]:
# 2. Load parquet
parquet_path = data_dir / "BTC-USDT-SWAP" / "grid100ms" / "2026-03-04" / (
    "book_grid100ms_BTC-USDT-SWAP_2026-03-04_10-00-00__2026-03-04_11-00-00.parquet"
)
if not parquet_path.exists():
    found = sorted(data_dir.rglob("*.parquet"))
    parquet_path = found[0] if found else None
    if found:
        print("Default parquet not found, loading:", parquet_path)
if parquet_path is None or not Path(parquet_path).exists():
    raise FileNotFoundError(
        f"No parquet found. Check data_dir={data_dir}. "
        "Run reconstruct_orderbook_to_parquet.py first."
    )
df = pd.read_parquet(parquet_path)
df.head()


In [ ]:
# 1 hour slice
_df = df.copy()
_df["ts_event"] = pd.to_datetime(_df["ts_event"], utc=True)
_df = _df.sort_values("ts_event").reset_index(drop=True)
start_ts = _df["ts_event"].min()
end_ts = start_ts + pd.Timedelta(hours=1)
_df = _df[_df["ts_event"].between(start_ts, end_ts)].reset_index(drop=True)
print("Rows:", len(_df))
print("Range:", _df["ts_event"].min(), "..", _df["ts_event"].max())

In [ ]:
# 2. Compute features
_df["mid_price"] = (_df["bid_px_01"] + _df["ask_px_01"]) / 2
_df["spread"] = _df["ask_px_01"] - _df["bid_px_01"]

# imbalance L1
_denom = (_df["bid_sz_01"] + _df["ask_sz_01"]).replace(0, np.nan)
_df["imbalance_l1"] = ((_df["bid_sz_01"] - _df["ask_sz_01"]) / _denom).fillna(0)

# microprice diff
_denom_mp = (_df["bid_sz_01"] + _df["ask_sz_01"]).replace(0, np.nan)
_df["microprice"] = (
    (_df["ask_px_01"] * _df["bid_sz_01"]) + (_df["bid_px_01"] * _df["ask_sz_01"])
) / _denom_mp
_df["microprice_diff"] = (_df["microprice"] - _df["mid_price"]).fillna(0)

# order flow imbalance (approx, from L1 size changes)
_df["delta_bid_sz_01"] = _df["bid_sz_01"].diff().fillna(0)
_df["delta_ask_sz_01"] = _df["ask_sz_01"].diff().fillna(0)
buy_flow = _df["delta_bid_sz_01"].clip(lower=0) + (-_df["delta_ask_sz_01"]).clip(lower=0)
sell_flow = _df["delta_ask_sz_01"].clip(lower=0) + (-_df["delta_bid_sz_01"]).clip(lower=0)
_denom_of = (buy_flow + sell_flow).replace(0, np.nan)
_df["order_flow_imbalance"] = ((buy_flow - sell_flow) / _denom_of).fillna(0)

_df[["imbalance_l1", "spread", "microprice_diff", "order_flow_imbalance"]].head()

In [ ]:
# 3. Target: 200ms future mid
_df["future_mid"] = _df["mid_price"].shift(-2)
_df["delta"] = _df["future_mid"] - _df["mid_price"]
_df["target"] = np.sign(_df["delta"]).astype("Int64")

_df2 = _df.dropna(subset=["future_mid"]).reset_index(drop=True)
print("Rows after shift:", len(_df2))

In [ ]:
# 4. Basic statistics
for col in ["imbalance_l1", "spread", "microprice_diff", "order_flow_imbalance"]:
    s = _df2[col].dropna()
    print(f"\n{col}")
    print("mean:", s.mean())
    print("std:", s.std())
    print("min/max:", s.min(), s.max())

# 5. Distribution plots
for col in ["imbalance_l1", "spread", "microprice_diff", "order_flow_imbalance"]:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(_df2[col].values, bins=100)
    ax.set_title(f"{col} histogram")
    ax.set_xlabel(col)
    ax.set_ylabel("count")
    plt.tight_layout()
    plt.show()

In [ ]:
# 6. Relationship to future price (2D binning)

def plot_2d_mean_target(df_in: pd.DataFrame, x: str, y: str, q: int = 5) -> None:
    d = df_in[[x, y, "target"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(d) < 10:
        print(f"Skip {x} vs {y}: too few rows ({len(d)})")
        return
    try:
        d["x_bin"] = pd.qcut(d[x], q=q, duplicates="drop")
        d["y_bin"] = pd.qcut(d[y], q=q, duplicates="drop")
    except ValueError as e:
        print(f"Skip {x} vs {y}: qcut failed ({e})")
        return

    pivot = d.pivot_table(index="y_bin", columns="x_bin", values="target", aggfunc="mean")
    if pivot.empty or pivot.size < 2:
        print(f"Skip {x} vs {y}: pivot too small")
        return

    vals = np.nan_to_num(pivot.values, nan=0.0)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(vals, aspect="auto", interpolation="nearest")
    ax.set_title(f"mean(target) by {y} vs {x} (q={q})")
    ax.set_xlabel(f"{x} bins")
    ax.set_ylabel(f"{y} bins")
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels([str(c) for c in pivot.columns], rotation=90)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels([str(i) for i in pivot.index])
    plt.colorbar(im, ax=ax, label="mean(target)")
    plt.tight_layout()
    plt.show()

plot_2d_mean_target(_df2, x="imbalance_l1", y="spread", q=5)
plot_2d_mean_target(_df2, x="imbalance_l1", y="microprice_diff", q=5)
plot_2d_mean_target(_df2, x="imbalance_l1", y="order_flow_imbalance", q=5)

# 7. Time series (downsample)
down = _df2.iloc[::10].copy()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(down["ts_event"], down["imbalance_l1"].values)
ax.set_title("imbalance_l1 over time (downsampled)")
plt.tight_layout()
plt.show()

# 8. Correlation analysis
for col in ["imbalance_l1", "spread", "microprice_diff", "order_flow_imbalance"]:
    print(f"Correlation {col} vs delta:", _df2[col].corr(_df2["delta"]))

# 9. Extreme events analysis: top 1% by |imbalance_l1|
thr = _df2["imbalance_l1"].abs().quantile(0.99)
ext = _df2[_df2["imbalance_l1"].abs() >= thr]
print("\nExtreme events |imbalance_l1| >= 99% quantile")
print("threshold:", thr)
print("count:", len(ext))
print("avg delta:", ext["delta"].mean())
print("mean target:", ext["target"].mean())